<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>01 — تثبيت مكتبات التدريب</h2>

  <p>
    تثبّت هذه الخلية المكتبات الأساسية المطلوبة لتدريب نموذج
    <span dir="ltr">SentenceTransformer</span>
    داخل بيئة
    <span dir="ltr">Google Colab</span>.
  </p>

  <p>
    نستخدم مكتبة
    <span dir="ltr">sentence-transformers</span>
    لتحميل نموذج
    <span dir="ltr">MiniLM</span>
    وتدريبه، ومكتبة
    <span dir="ltr">datasets</span>
    لتحويل أزواج التدريب إلى صيغة مناسبة للتدريب، ومكتبة
    <span dir="ltr">accelerate</span>
    لتحسين تشغيل التدريب على
    <span dir="ltr">GPU</span>، ومكتبة
    <span dir="ltr">transformers</span>
    كبنية أساسية للنموذج.
  </p>

  <p>
    الخيار
    <span dir="ltr">-U</span>
    يحدّث المكتبات إلى أحدث نسخة متاحة، بينما الخيار
    <span dir="ltr">-q</span>
    يجعل التثبيت أقل إظهارًا للتفاصيل داخل المخرجات.
  </p>
</div>

In [ ]:
!pip install -q -U sentence-transformers datasets accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 14.2 MB/s eta 0:00:00


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>02 — ربط Google Drive</h2>

  <p>
    تربط هذه الخلية
    <span dir="ltr">Google Colab</span>
    مع
    <span dir="ltr">Google Drive</span>
    للوصول إلى ملفات المشروع، خصوصًا ملفات أزواج التدريب والتحقق المحفوظة مسبقًا.
  </p>

  <p>
    <strong>إشارة التحقق:</strong>
    الخلية الصحيحة تحتوي على
    <span dir="ltr">drive.mount("/content/drive")</span>.
  </p>
</div>

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>03 — إعداد البيئة والمسارات الأساسية للتدريب</h2>

  <p>
    تجهّز هذه الخلية بيئة التدريب: تستورد المكتبات، تثبّت
    <span dir="ltr">random seed</span>
    لضمان قابلية إعادة الإنتاج، وتحدد مسارات أزواج التدريب والتحقق ومجلد حفظ النموذج الناتج.
  </p>

  <p>
    كما تتحقق من وجود ملفات
    <span dir="ltr">1.5M train pairs</span>
    و
    <span dir="ltr">30k validation pairs</span>
    وتعرض حالة
    <span dir="ltr">CUDA/GPU</span>
    للتأكد أن التدريب سيعمل على العتاد المناسب.
  </p>

  <p>
    <strong>إشارة التحقق:</strong>
    الخلية الصحيحة تحتوي على
    <span dir="ltr">TRAIN_PAIRS_PATH</span>،
    <span dir="ltr">VAL_PAIRS_PATH</span>،
    <span dir="ltr">RUN_NAME</span>
    و
    <span dir="ltr">torch.cuda.is_available()</span>.
  </p>
</div>

In [ ]:
from pathlib import Path
import os
import json
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from sentence_transformers import SentenceTransformer
from sentence_transformers import losses
from sentence_transformers.evaluation import InformationRetrievalEvaluator
from sentence_transformers.trainer import SentenceTransformerTrainer
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

RANDOM_STATE = 42

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

PROJECT_ROOT = Path("/content/drive/MyDrive/semester project/news_comment_topic_system")

PAIR_DIR = PROJECT_ROOT / "training_data/v7_raw_pair_mining_20260428_232654"

TRAIN_PAIRS_PATH = PAIR_DIR / "train_stage1_pairs_v7_raw_large_balanced_1p5M.csv"
VAL_PAIRS_PATH = PAIR_DIR / "val_stage1_pairs_v7_raw_large_balanced_30k.csv"
PAIR_METADATA_PATH = PAIR_DIR / "v7_raw_large_balanced_pair_mining_metadata.json"

MODELS_DIR = PROJECT_ROOT / "models"
RUN_NAME = "v7_fullft_minilm_raw_largepairs_1p5M_seq256"

OUTPUT_DIR = MODELS_DIR / RUN_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("TRAIN_PAIRS_PATH exists:", TRAIN_PAIRS_PATH.exists())
print("VAL_PAIRS_PATH exists:", VAL_PAIRS_PATH.exists())
print("PAIR_METADATA_PATH exists:", PAIR_METADATA_PATH.exists())
print("OUTPUT_DIR:", OUTPUT_DIR)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU RAM GB:", round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2))

/tmp/ipykernel_1343/392708083.py:11: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses
/tmp/ipykernel_1343/392708083.py:12: DeprecationWarning: Importing from 'sentence_transformers.evaluation' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.evaluation' instead.
  from sentence_transformers.evaluation import InformationRetrievalEvaluator
/tmp/ipykernel_1343/392708083.py:13: DeprecationWarning: Importing from 'sentence_transformers.trainer' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.trainer' instead.
  from sentence_transformers.trainer import SentenceTransformerTrainer
/tmp/ipykernel_1343/392708083.py:14: DeprecationWarning: Importing from 'sentence_transformers.training_args'

TRAIN_PAIRS_PATH exists: True
VAL_PAIRS_PATH exists: True
PAIR_METADATA_PATH exists: True
OUTPUT_DIR: /content/drive/MyDrive/semester project/news_comment_topic_system/models/v7_fullft_minilm_raw_largepairs_1p5M_seq256
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
GPU RAM GB: 39.49


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>04 — تحميل أزواج التدريب والتحقق بصيغ ثابتة</h2>

  <p>
    تحمّل هذه الخلية ملفات أزواج التدريب والتحقق مع تثبيت أنواع الأعمدة الأساسية، ثم تحوّل
    <span dir="ltr">similarity</span>
    إلى قيمة رقمية لاستخدامها في الفحص والتوثيق.
  </p>

  <p>
    تعرض الخلية حجم البيانات، أنواع الأعمدة، عينات من الأزواج، وإحصاءات التشابه للتأكد من جاهزية البيانات قبل تدريب النموذج.
  </p>

  <p>
    <strong>إشارة التحقق:</strong>
    تحتوي الخلية على
    <span dir="ltr">dtype={...}</span>،
    <span dir="ltr">low_memory=False</span>
    و
    <span dir="ltr">similarity.astype(float)</span>.
  </p>
</div>

In [ ]:
train_pairs = pd.read_csv(
    TRAIN_PAIRS_PATH,
    dtype={
        "post_id_true": str,
        "text_a": str,
        "text_b": str,
        "split": str,
    },
    low_memory=False
)

val_pairs = pd.read_csv(
    VAL_PAIRS_PATH,
    dtype={
        "post_id_true": str,
        "text_a": str,
        "text_b": str,
        "split": str,
    },
    low_memory=False
)

train_pairs["similarity"] = train_pairs["similarity"].astype(float)
val_pairs["similarity"] = val_pairs["similarity"].astype(float)

print("Train pairs:", train_pairs.shape)
print("Val pairs:", val_pairs.shape)

print(train_pairs.dtypes)

display(train_pairs.head())
display(val_pairs.head())

print("\nTrain similarity:")
display(train_pairs["similarity"].describe())

print("\nVal similarity:")
display(val_pairs["similarity"].describe())

<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>05 — التحقق من صلاحية أزواج التدريب والتحقق</h2>

  <p>
    تتحقق هذه الخلية من وجود الأعمدة الأساسية:
    <span dir="ltr">text_a</span>،
    <span dir="ltr">text_b</span>،
    و
    <span dir="ltr">similarity</span>.
  </p>

  <p>
    ثم تنظف القيم الفارغة وتستبعد الأزواج ذات النصوص القصيرة جدًا، لضمان أن بيانات
    <span dir="ltr">fine-tuning</span>
    تحتوي على أزواج نصية صالحة.
  </p>
</div>

In [ ]:
REQUIRED_COLUMNS = ["text_a", "text_b", "similarity"]

for col in REQUIRED_COLUMNS:
    if col not in train_pairs.columns:
        raise ValueError(f"Missing column in train_pairs: {col}")
    if col not in val_pairs.columns:
        raise ValueError(f"Missing column in val_pairs: {col}")

train_pairs["text_a"] = train_pairs["text_a"].fillna("").astype(str)
train_pairs["text_b"] = train_pairs["text_b"].fillna("").astype(str)

val_pairs["text_a"] = val_pairs["text_a"].fillna("").astype(str)
val_pairs["text_b"] = val_pairs["text_b"].fillna("").astype(str)

before_train = len(train_pairs)
before_val = len(val_pairs)

train_pairs = train_pairs[
    (train_pairs["text_a"].str.len() > 2) &
    (train_pairs["text_b"].str.len() > 2)
].reset_index(drop=True)

val_pairs = val_pairs[
    (val_pairs["text_a"].str.len() > 2) &
    (val_pairs["text_b"].str.len() > 2)
].reset_index(drop=True)

print("Train before:", before_train)
print("Train after:", len(train_pairs))

print("Val before:", before_val)
print("Val after:", len(val_pairs))

Train before: 1500000
Train after: 1500000
Val before: 30000
Val after: 30000


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>06 — تحميل النموذج الأساسي</h2>

  <p>
    تحمّل هذه الخلية نموذج
    <span dir="ltr">paraphrase-multilingual-MiniLM-L12-v2</span>
    كنقطة بداية لعملية
    <span dir="ltr">fine-tuning</span>.
  </p>

  <p>
    يتم ضبط طول التسلسل على
    <span dir="ltr">256</span>
    واختيار الجهاز المتاح تلقائيًا، سواء
    <span dir="ltr">GPU</span>
    أو
    <span dir="ltr">CPU</span>.
  </p>
</div>

In [ ]:
BASE_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
MAX_SEQ_LENGTH = 256

device = "cuda" if torch.cuda.is_available() else "cpu"

model = SentenceTransformer(BASE_MODEL_NAME, device=device)
model.max_seq_length = MAX_SEQ_LENGTH

print("Loaded base model:", BASE_MODEL_NAME)
print("Device:", device)
print("Max sequence length:", model.max_seq_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/526 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded base model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Device: cuda
Max sequence length: 256


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>07 — تفعيل Full Fine-tuning</h2>

  <p>
    تجعل هذه الخلية جميع معاملات النموذج قابلة للتحديث أثناء التدريب عبر
    <span dir="ltr">requires_grad=True</span>،
    وهذا يعني أن التدريب سيكون
    <span dir="ltr">Full Fine-tuning</span>
    وليس تدريب طبقة أخيرة فقط.
  </p>

  <p>
    تعرض الخلية عدد المعاملات الكلي وعدد المعاملات القابلة للتدريب للتأكد أن نسبة التدريب تساوي تقريبًا
    <span dir="ltr">100%</span>.
  </p>
</div>

In [ ]:
for param in model.parameters():
    param.requires_grad = True

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable ratio:", round(trainable_params / total_params * 100, 2), "%")

Total parameters: 117653760
Trainable parameters: 117653760
Trainable ratio: 100.0 %


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>08 — تحويل الأزواج إلى Dataset للتدريب</h2>

  <p>
    تحوّل هذه الخلية أزواج
    <span dir="ltr">text_a</span>
    و
    <span dir="ltr">text_b</span>
    إلى صيغة
    <span dir="ltr">Hugging Face Dataset</span>
    المناسبة لـ
    <span dir="ltr">SentenceTransformerTrainer</span>.
  </p>

  <p>
    يتم تغيير أسماء الأعمدة إلى
    <span dir="ltr">sentence1</span>
    و
    <span dir="ltr">sentence2</span>
    لأن كل صف يمثل زوجًا موجبًا يستخدمه النموذج أثناء
    <span dir="ltr">fine-tuning</span>.
  </p>
</div>

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_pairs[["text_a", "text_b"]].rename(
        columns={"text_a": "sentence1", "text_b": "sentence2"}
    ),
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_pairs[["text_a", "text_b"]].rename(
        columns={"text_a": "sentence1", "text_b": "sentence2"}
    ),
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(train_dataset[0])

Dataset({
    features: ['sentence1', 'sentence2'],
    num_rows: 1500000
})
Dataset({
    features: ['sentence1', 'sentence2'],
    num_rows: 30000
})
{'sentence1': 'What a waste of their appearance. Their oath is to the Constitution not to DT. Sleazy two more men as well as Rosenstein. They all need to to be dumped. No reason they could not answer question. I call BS on them. Smart questioners Angus King and our Senator Kamala Harris. Kudos to them for calling out these yokels.', 'sentence2': "Looks like contempt. Apparently these civil servants do not operate within the parameters of their oaths and the law. They don't answer to the representatives of the American people, just to one person. The Republic has been pushed aside for the dictatorship."}


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>09 — بناء مقيم الاسترجاع على بيانات التحقق</h2>

  <p>
    تبني هذه الخلية مقيّم
    <span dir="ltr">InformationRetrievalEvaluator</span>
    باستخدام عينة من أزواج التحقق، حيث يُعامل
    <span dir="ltr">text_a</span>
    كاستعلام و
    <span dir="ltr">text_b</span>
    كالنص الصحيح المطلوب استرجاعه.
  </p>

  <p>
    يستخدم هذا المقيّم لمراقبة قدرة النموذج أثناء التدريب على تقريب الأزواج الصحيحة دلاليًا، عبر مقاييس مثل
    <span dir="ltr">Accuracy@K</span>،
    <span dir="ltr">MRR@10</span>،
    <span dir="ltr">NDCG@10</span>
    و
    <span dir="ltr">MAP@100</span>.
  </p>
</div>

In [ ]:
from sentence_transformers.evaluation import InformationRetrievalEvaluator

EVAL_PAIRS_FOR_IR = 5000

val_eval_pairs = val_pairs.sample(
    n=min(EVAL_PAIRS_FOR_IR, len(val_pairs)),
    random_state=RANDOM_STATE
).reset_index(drop=True)

queries = {}
corpus = {}
relevant_docs = {}

for i, row in val_eval_pairs.iterrows():
    qid = f"q{i}"
    cid = f"c{i}"

    queries[qid] = row["text_a"]
    corpus[cid] = row["text_b"]
    relevant_docs[qid] = {cid}

ir_evaluator = InformationRetrievalEvaluator(
    queries=queries,
    corpus=corpus,
    relevant_docs=relevant_docs,
    name="val_ir_v7_raw_largepairs",
    show_progress_bar=True,
    mrr_at_k=[10],
    ndcg_at_k=[10],
    accuracy_at_k=[1, 3, 5, 10],
    precision_recall_at_k=[1, 3, 5, 10],
    map_at_k=[100],
)

print("IR evaluator queries:", len(queries))
print("IR evaluator corpus:", len(corpus))

IR evaluator queries: 5000
IR evaluator corpus: 5000


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>10 — تعريف دالة الخسارة</h2>

  <p>
    تعرّف هذه الخلية دالة الخسارة
    <span dir="ltr">MultipleNegativesRankingLoss</span>
    المستخدمة لتدريب النموذج على تقريب الأزواج الصحيحة
    <span dir="ltr">text_a</span>
    و
    <span dir="ltr">text_b</span>.
  </p>

  <p>
    تعتمد هذه الدالة على اعتبار الأزواج الأخرى داخل نفس
    <span dir="ltr">batch</span>
    كأمثلة سالبة ضمنية، مما يجعلها مناسبة جدًا لتدريب نماذج
    <span dir="ltr">SentenceTransformer</span>
    على أزواج دلالية.
  </p>
</div>

In [ ]:
from sentence_transformers import losses

train_loss = losses.MultipleNegativesRankingLoss(model)

print("Loss:", train_loss)

Loss: MultipleNegativesRankingLoss(
  (model): SentenceTransformer(
    (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
    (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  )
)


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>11 — ضبط Hyperparameters التدريب</h2>

  <p>
    تحدد هذه الخلية إعدادات تدريب نموذج
    <span dir="ltr">SentenceTransformer</span>،
    مثل عدد العصور
    <span dir="ltr">epochs</span>،
    حجم الدفعة
    <span dir="ltr">batch size</span>،
    معدل التعلم
    <span dir="ltr">learning rate</span>،
    وخطوات التقييم والحفظ.
  </p>

  <p>
    نستخدم
    <span dir="ltr">NO_DUPLICATES</span>
    حتى لا تتكرر النصوص داخل الدفعة، وهذا مهم مع
    <span dir="ltr">MultipleNegativesRankingLoss</span>
    لأن عناصر الدفعة تُستخدم كأمثلة سالبة ضمنية.
  </p>
</div>

In [ ]:
from sentence_transformers.training_args import SentenceTransformerTrainingArguments, BatchSamplers

training_args = SentenceTransformerTrainingArguments(
    output_dir=str(OUTPUT_DIR),

    num_train_epochs=1,

    per_device_train_batch_size=256,
    per_device_eval_batch_size=256,######192

    learning_rate=1e-5,
    warmup_ratio=0.10,
    weight_decay=0.01,

    bf16=True,
    fp16=False,

    batch_sampler=BatchSamplers.NO_DUPLICATES,

    logging_steps=200,

    eval_strategy="steps",
    eval_steps=2000,

    save_strategy="steps",
    save_steps=2000,
    save_total_limit=2,

    load_best_model_at_end=False,

    run_name=RUN_NAME,
    seed=RANDOM_STATE,

    dataloader_num_workers=2,

    report_to="none"
)

print(training_args)

SentenceTransformerTrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
batch_sampler=BatchSamplers.NO_DUPLICATES,
bf16=True,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=2,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=False,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=2000,
eva

<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>12 — بناء كائن التدريب Trainer</h2>

  <p>
    تنشئ هذه الخلية كائن
    <span dir="ltr">SentenceTransformerTrainer</span>
    الذي يجمع النموذج، بيانات التدريب، بيانات التحقق، دالة الخسارة، ومقيّم الاسترجاع في وحدة تدريب واحدة.
  </p>

  <p>
    بعد هذه الخلية يصبح النظام جاهزًا لبدء
    <span dir="ltr">fine-tuning</span>
    باستخدام الإعدادات المحددة في الخلية السابقة.
  </p>
</div>

In [ ]:
from sentence_transformers.trainer import SentenceTransformerTrainer

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    loss=train_loss,
    evaluator=ir_evaluator,
)

print("Trainer is ready.")

Computing widget examples:   0%|          | 0/1 [00:00<?, ?example/s]

Trainer is ready.


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>13 — بدء تدريب النموذج</h2>

  <p>
    تشغّل هذه الخلية عملية
    <span dir="ltr">Full Fine-tuning</span>
    للنموذج على أزواج التعليقات باستخدام
    <span dir="ltr">MultipleNegativesRankingLoss</span>.
  </p>

  <p>
    أثناء التدريب يتم تسجيل
    <span dir="ltr">training loss</span>
    وتنفيذ تقييم دوري على بيانات التحقق باستخدام مقاييس الاسترجاع مثل
    <span dir="ltr">Accuracy@K</span> و
    <span dir="ltr">MRR@10</span>
    و
    <span dir="ltr">MAP@100</span>.
  </p>
</div>

In [ ]:
train_result = trainer.train()
print(train_result)

Step,Training Loss,Validation Loss,Val Ir V7 Raw Largepairs Cosine Accuracy@1,Val Ir V7 Raw Largepairs Cosine Accuracy@3,Val Ir V7 Raw Largepairs Cosine Accuracy@5,Val Ir V7 Raw Largepairs Cosine Accuracy@10,Val Ir V7 Raw Largepairs Cosine Precision@1,Val Ir V7 Raw Largepairs Cosine Precision@3,Val Ir V7 Raw Largepairs Cosine Precision@5,Val Ir V7 Raw Largepairs Cosine Precision@10,Val Ir V7 Raw Largepairs Cosine Recall@1,Val Ir V7 Raw Largepairs Cosine Recall@3,Val Ir V7 Raw Largepairs Cosine Recall@5,Val Ir V7 Raw Largepairs Cosine Recall@10,Val Ir V7 Raw Largepairs Cosine Ndcg@10,Val Ir V7 Raw Largepairs Cosine Mrr@10,Val Ir V7 Raw Largepairs Cosine Map@100
2000,1.598011,1.676859,0.157200,0.311200,0.400600,0.519800,0.157200,0.103733,0.080120,0.051980,0.157200,0.311200,0.400600,0.519800,0.321173,0.259641,0.272846
4000,1.570705,1.649383,0.160000,0.311000,0.404400,0.523000,0.160000,0.103667,0.080880,0.052300,0.160000,0.311000,0.404400,0.523000,0.323913,0.262221,0.275758


Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Corpus Chunks:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:02<00:00,  2.90s/it]


...

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

Corpus Chunks: 100%|██████████| 1/1 [00:02<00:00,  2.86s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=5860, training_loss=1.6162913612131373, metrics={'train_runtime': 2279.3356, 'train_samples_per_second': 658.086, 'train_steps_per_second': 2.571, 'total_flos': 0.0, 'train_loss': 1.6162913612131373, 'epoch': 1.0})


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>14 — حفظ النموذج النهائي</h2>

  <p>
    تحفظ هذه الخلية النموذج بعد انتهاء عملية
    <span dir="ltr">fine-tuning</span>
    داخل مجلد
    <span dir="ltr">final_model</span>.
  </p>

  <p>
    هذا المجلد هو النسخة النهائية التي سيتم تحميلها لاحقًا لتوليد
    <span dir="ltr">embeddings</span>
    واستخدامها داخل
    <span dir="ltr">BERTopic</span>.
  </p>
</div>

In [ ]:
FINAL_MODEL_DIR = OUTPUT_DIR / "final_model"
model.save_pretrained(str(FINAL_MODEL_DIR))

print("Saved final model:", FINAL_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved final model: /content/drive/MyDrive/semester project/news_comment_topic_system/models/v7_fullft_minilm_raw_largepairs_1p5M_seq256/final_model


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>15 — حفظ Metadata التدريب</h2>

  <p>
    تحفظ هذه الخلية ملف
    <span dir="ltr">training_metadata.json</span>
    لتوثيق إعدادات التدريب ونتائجه الأساسية، مثل نوع التدريب، عدد الأزواج، حجم
    <span dir="ltr">batch</span>،
    معدل التعلم، عدد الخطوات، وقيم التقييم أثناء التدريب.
  </p>

  <p>
    هذا الملف مهم لإعادة إنتاج التجربة لاحقًا، ولتوضيح أن النموذج دُرّب باستخدام
    <span dir="ltr">Full Fine-tuning</span>
    و
    <span dir="ltr">MultipleNegativesRankingLoss</span>
    على أزواج تعليقات دلالية.
  </p>
</div>

In [ ]:
import json

training_metadata = {
    "experiment_name": RUN_NAME,
    "base_model": BASE_MODEL_NAME,
    "final_model_dir": str(FINAL_MODEL_DIR),

    "training_type": "full_fine_tuning",
    "all_parameters_trainable": True,
    "total_parameters": int(total_params),
    "trainable_parameters": int(trainable_params),
    "trainable_ratio_percent": float(round(trainable_params / total_params * 100, 2)),

    "train_pairs_path": str(TRAIN_PAIRS_PATH),
    "val_pairs_path": str(VAL_PAIRS_PATH),
    "pair_metadata_path": str(PAIR_METADATA_PATH),

    "train_pairs": int(len(train_pairs)),
    "val_pairs": int(len(val_pairs)),
    "eval_pairs_for_ir": int(len(val_eval_pairs)),

    "max_seq_length": MAX_SEQ_LENGTH,
    "loss": "MultipleNegativesRankingLoss",
    "batch_sampler": "NO_DUPLICATES",

    "num_train_epochs": 1,
    "global_steps": 5860,

    "per_device_train_batch_size": 256,
    "per_device_eval_batch_size": 256,

    "learning_rate": 1e-5,
    "warmup_ratio": 0.10,
    "weight_decay": 0.01,
    "bf16": True,
    "fp16": False,
    "random_state": RANDOM_STATE,

    "train_runtime_seconds": 2279.3356,
    "train_samples_per_second": 658.086,
    "train_steps_per_second": 2.571,
    "average_train_loss_full_run": 1.6162913612131373,

    "step_2000": {
        "training_loss": 1.598011,
        "validation_loss": 1.676859,
        "accuracy_at_1": 0.157200,
        "accuracy_at_3": 0.311200,
        "accuracy_at_5": 0.400600,
        "accuracy_at_10": 0.519800,
        "ndcg_at_10": 0.321173,
        "mrr_at_10": 0.259641,
        "map_at_100": 0.272846,
    },

    "step_4000": {
        "training_loss": 1.570705,
        "validation_loss": 1.649383,
        "accuracy_at_1": 0.160000,
        "accuracy_at_3": 0.311000,
        "accuracy_at_5": 0.404400,
        "accuracy_at_10": 0.523000,
        "ndcg_at_10": 0.323913,
        "mrr_at_10": 0.262221,
        "map_at_100": 0.275758,
    }
}

METADATA_OUT = OUTPUT_DIR / "training_metadata.json"

with open(METADATA_OUT, "w", encoding="utf-8") as f:
    json.dump(training_metadata, f, indent=2, ensure_ascii=False)

print("Saved metadata:", METADATA_OUT)

Saved metadata: /content/drive/MyDrive/semester project/news_comment_topic_system/models/v7_fullft_minilm_raw_largepairs_1p5M_seq256/training_metadata.json


<div dir="rtl" style="text-align:right; line-height:1.8;">
  <h2>16 — فحص النموذج بعد التدريب</h2>

  <p>
    تفحص هذه الخلية جودة النموذج النهائي بمقارنة تشابه الأزواج الصحيحة من بيانات التحقق مع أزواج عشوائية.
  </p>

  <p>
    إذا كان التدريب ناجحًا، يجب أن يكون متوسط
    <span dir="ltr">cosine similarity</span>
    للأزواج الصحيحة أعلى بوضوح من الأزواج العشوائية، مع فجوة واضحة
    <span dir="ltr">Positive-Random Mean Gap</span>.
  </p>
</div>

In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

trained_model = SentenceTransformer(str(FINAL_MODEL_DIR), device=device)
trained_model.max_seq_length = MAX_SEQ_LENGTH

sample_pairs = val_pairs.sample(n=1000, random_state=RANDOM_STATE).reset_index(drop=True)

emb_a = trained_model.encode(
    sample_pairs["text_a"].tolist(),
    batch_size=256,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

emb_b = trained_model.encode(
    sample_pairs["text_b"].tolist(),
    batch_size=256,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

positive_scores = util.cos_sim(emb_a, emb_b).diagonal().detach().cpu().numpy()

random_b = sample_pairs["text_b"].sample(frac=1.0, random_state=RANDOM_STATE).tolist()

emb_random_b = trained_model.encode(
    random_b,
    batch_size=256,
    convert_to_tensor=True,
    normalize_embeddings=True,
    show_progress_bar=True
)

random_scores = util.cos_sim(emb_a, emb_random_b).diagonal().detach().cpu().numpy()

print("Average positive-pair cosine:", round(float(np.mean(positive_scores)), 4))
print("Median positive-pair cosine:", round(float(np.median(positive_scores)), 4))
print("Min positive-pair cosine:", round(float(np.min(positive_scores)), 4))
print("Max positive-pair cosine:", round(float(np.max(positive_scores)), 4))

print("\nAverage random-pair cosine:", round(float(np.mean(random_scores)), 4))
print("Median random-pair cosine:", round(float(np.median(random_scores)), 4))
print("Min random-pair cosine:", round(float(np.min(random_scores)), 4))
print("Max random-pair cosine:", round(float(np.max(random_scores)), 4))

print("\nPositive-Random Mean Gap:", round(float(np.mean(positive_scores) - np.mean(random_scores)), 4))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Batches:   0%|          | 0/4 [00:00<?, ?it/s]

Average positive-pair cosine: 0.5556
Median positive-pair cosine: 0.5529
Min positive-pair cosine: 0.2404
Max positive-pair cosine: 0.916

Average random-pair cosine: 0.0728
Median random-pair cosine: 0.0559
Min random-pair cosine: -0.3001
Max random-pair cosine: 0.629

Positive-Random Mean Gap: 0.4828
